# 05 · Tool use

**Goal:** give Claude functions it can ask you to run, so it can work with live data and take actions, and build the loop that makes this work.

| | |
|---|---|
| **You will learn** | Tool functions and schemas, the request-and-result round trip, the tool loop, adding tools, making Claude use a tool, streaming tool arguments, and the SDK's tool runner |
| **Prerequisites** | [Notebook 01](01_conversations_and_control.ipynb), for keeping conversation history |
| **Run time** | About 2 minutes |
| **Cost** | About $0.07 when I ran it. This notebook stops itself at $0.25 |
| **Paid cells** | Sections 2, 3, 4, 6 and 7. Sections 1 and 5 are free |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## Why tools?

Claude only knows what it learned in training. It can't check today's date, read your database or set a reminder. Tools fix that.

Think of Claude as a brilliant consultant in a room with no phone or computer. Claude can't look anything up itself, but it can pass you a note: *"Please run `get_current_datetime` and tell me the result."* You run it, pass the answer back, and Claude carries on.

```
 1. You send:     the question + descriptions of the tools Claude may use
 2. Claude sends: a tool_use request ("run get_current_datetime")
 3. You run the function yourself, on your machine
 4. You send:     a tool_result with the output
 5. Claude sends: the final answer, or asks for another tool
```

**Claude never runs your code.** It only asks. Your program decides whether to run a tool, and it does the running.

## Setup

The same setup as the earlier notebooks, plus the conversation helpers from notebook 01. Run the cells from top to bottom.

In [6]:
import json
import os
import time
from datetime import datetime, timedelta

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


In [7]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.25     # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, response):
    messages.append({"role": "assistant", "content": response.content})

## 1. A tool is a function plus a schema (free)

Every tool you write has two parts:

1. **The function.** Plain Python that does the work. Claude never sees this code.
2. **The schema.** A description of the tool: its `name`, a `description` saying what it does **and when to use it**, and an `input_schema` (JSON Schema) for its arguments. This is all Claude sees.

A **dispatcher**, `run_tool`, maps the tool name Claude asks for onto the right function.

In [8]:
def add(a, b):
    return a + b


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(date_format)


add_schema = {
    "name": "add",
    "description": "Add two numbers and return the sum. Use this for any addition, instead of working it out yourself.",
    "input_schema": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "The first number"},
            "b": {"type": "number", "description": "The second number"},
        },
        "required": ["a", "b"],
    },
}

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Get the current date and time. Use this whenever the answer depends on today's date. "
                   "Returns a string like '2026-10-02 14:05:00'.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {"type": "string", "description": "strftime pattern. Defaults to '%Y-%m-%d %H:%M:%S'"},
        },
        "required": [],
    },
}

tools = [add_schema, get_current_datetime_schema]


def run_tool(tool_name, tool_input):
    """Route a tool request from Claude to the matching Python function."""
    if tool_name == "add":
        return add(**tool_input)
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


print("Local test, no API call:", run_tool("add", {"a": 2, "b": 3}))

Local test, no API call: 5


**Tool definitions cost tokens.** Schemas are sent with every request, along with instructions telling Claude how to use tools. The free `count_tokens` endpoint shows how much that adds:

In [9]:
question = [{"role": "user", "content": "What is 10 + 10?"}]

without_tools = client.messages.count_tokens(model=MODEL, messages=question).input_tokens
with_tools = client.messages.count_tokens(model=MODEL, messages=question, tools=tools).input_tokens

print(f"Without tools: {without_tools:,} input tokens")
print(f"With 2 tools : {with_tools:,} input tokens")
print(f"Every request with these tools costs {with_tools - without_tools:,} extra input tokens.")

assert with_tools > without_tools

Without tools: 17 input tokens
With 2 tools : 568 input tokens
Every request with these tools costs 551 extra input tokens.


## 2. One round trip, by hand (paid)

We ask a question with two parts, so Claude can request **two tool calls in one response**. That's called parallel tool use.

Each `tool_use` block has a unique `id`. The two `add` requests look identical apart from their numbers, so the **`id` is how Claude matches each result to the request it answers**, even if you return them in a different order.

**Rules for sending results back:**

- Put a `tool_result` for **every** `tool_use` in **one** user message.
- Give each `tool_result` the `tool_use_id` of the request it answers.
- Send `tools` again with the next request. Claude needs the schemas to read the history.

In [10]:
messages = []
add_user_message(messages, "What is 10 + 10, and what is 30 + 30?")

response = track(client.messages.create(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=messages,
    tools=tools,
))

print("stop_reason:", response.stop_reason)
print("Block types:", [block.type for block in response.content])

tool_use_blocks = [block for block in response.content if block.type == "tool_use"]
for block in tool_use_blocks:
    print(f"   tool_use  id={block.id}  name={block.name}  input={block.input}")

assert response.stop_reason == "tool_use"
assert tool_use_blocks

[input=578 output=129 cost=$0.00245 | spent $0.0024 of $0.25]
stop_reason: tool_use
Block types: ['tool_use', 'tool_use']
   tool_use  id=toolu_01MRJuh7DoqGy2gR28ZBVXLH  name=add  input={'a': 10, 'b': 10}
   tool_use  id=toolu_01DK46n79u5ncMsEiBdUQhSx  name=add  input={'a': 30, 'b': 30}


In [11]:
# Keep Claude's whole turn in the history: its text AND every tool_use block.
add_assistant_message(messages, response)

tool_results = []
for block in tool_use_blocks:
    result = run_tool(block.name, block.input)
    print(f"Ran {block.name}({block.input}) = {result}   (answers id {block.id})")
    tool_results.append({
        "type": "tool_result",
        "tool_use_id": block.id,         # ties this result to its request
        "content": json.dumps(result),
    })

# Every result goes back together, in ONE user message.
add_user_message(messages, tool_results)

final = track(client.messages.create(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=messages,
    tools=tools,
))
answer = text_from_message(final)
print("\nFinal answer:", answer)

assert final.stop_reason == "end_turn"
assert "20" in answer and "60" in answer

Ran add({'a': 10, 'b': 10}) = 20   (answers id toolu_01MRJuh7DoqGy2gR28ZBVXLH)
Ran add({'a': 30, 'b': 30}) = 60   (answers id toolu_01DK46n79u5ncMsEiBdUQhSx)
[input=761 output=25 cost=$0.00177 | spent $0.0042 of $0.25]

Final answer: - 10 + 10 = 20
- 30 + 30 = 60


**What to notice**

- Claude got the arithmetic from your code, not from its own head. For sums this small it hardly matters, but for exact dates, prices or account balances it does.
- `stop_reason` was `tool_use` on the first response and `end_turn` on the last. **That's how your code knows whether Claude wants more tools.**

## 3. The tool loop (paid)

Section 2 handled one round of tools by hand. Some questions need tools **in sequence**, where one tool needs another's result. Take *"What day is 103 days from today?"*:

1. Claude calls `get_current_datetime` and gets today's date.
2. It reads that, then calls `add_duration_to_datetime` with it.
3. Now it can answer.

You can't know in advance how many rounds a question needs, so you write a **loop**: call Claude, and while `stop_reason` is `tool_use`, run the tools and send the results back. This loop is at the heart of every agent.

If a tool fails, we don't crash. We send the error back as a `tool_result` with `"is_error": True`, so Claude can read it and try something else.

In [22]:
def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d %H:%M:%S"):
    start = datetime.strptime(datetime_str, input_format)
    deltas = {
        "seconds": timedelta(seconds=duration),
        "minutes": timedelta(minutes=duration),
        "hours": timedelta(hours=duration),
        "days": timedelta(days=duration),
        "weeks": timedelta(weeks=duration),
    }
    return (start + deltas[unit]).strftime("%A, %B %d, %Y")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Add a duration to a date and time, and return the resulting date. "
                   "Use this for any date arithmetic. Pass datetime_str exactly as get_current_datetime returned it.",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {"type": "string", "description": "The starting date and time, as 'YYYY-MM-DD HH:MM:SS'"},
            "duration": {"type": "number", "description": "How much time to add"},
            "unit": {"type": "string", "enum": ["seconds", "minutes", "hours", "days", "weeks"]},
        },
        "required": ["datetime_str", "duration", "unit"],
    },
}

tools = [get_current_datetime_schema, add_duration_to_datetime_schema]


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    if tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(response):
    """Run every tool_use block in a response, and return the matching tool_result blocks."""
    results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        try:
            output = run_tool(block.name, block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)})
        except Exception as error:
            results.append({"type": "tool_result", "tool_use_id": block.id,
                            "content": f"Error: {error}", "is_error": True})
        print(f"      ran {block.name}({block.input}) -> {results[-1]['content']}")
    return results


def run_conversation(messages, tools, max_turns=8):
    """Call Claude, run any tools it asks for, and repeat until it answers."""
    for turn in range(1, max_turns + 1):
        response = track(client.messages.create(
            model=MODEL,
            max_tokens=1000,
            output_config={"effort": "low"},
            messages=messages,
            tools=tools,
        ))
        add_assistant_message(messages, response)
        requested = [block.name for block in response.content if block.type == "tool_use"]
        print(f"Turn {turn}: stop_reason={response.stop_reason}, tools requested: {requested or 'none'}")

        if response.stop_reason != "tool_use":
            print("\nFinal answer:", text_from_message(response))
            return response

        add_user_message(messages, run_tools(response))  # all results together

    raise RuntimeError(f"Stopped after {max_turns} turns without a final answer.")

In [13]:
messages = []
add_user_message(messages, "What day is 103 days from today?")
final = run_conversation(messages, tools)

expected = add_duration_to_datetime(get_current_datetime(), 103, "days")
print("\nChecked by our own code:", expected)

assert final.stop_reason == "end_turn"

[input=685 output=30 cost=$0.00167 | spent $0.0059 of $0.25]
Turn 1: stop_reason=tool_use, tools requested: ['get_current_datetime']
      ran get_current_datetime({}) -> "2026-11-12 10:07:46"
[input=734 output=113 cost=$0.00260 | spent $0.0085 of $0.25]
Turn 2: stop_reason=tool_use, tools requested: ['add_duration_to_datetime']
      ran add_duration_to_datetime({'datetime_str': '2026-11-12 10:07:46', 'duration': 103, 'unit': 'days'}) -> "Tuesday, February 23, 2027"
[input=867 output=33 cost=$0.00206 | spent $0.0105 of $0.25]
Turn 3: stop_reason=end_turn, tools requested: none

Final answer: 103 days from today (November 12, 2026) is **Tuesday, February 23, 2027**.

Checked by our own code: Tuesday, February 23, 2027


**What to notice**

- The loop ran more than once. Claude needed the result of the first tool before it could call the second, so these calls **chain** instead of running in parallel.
- `max_turns` is a safety limit. Without it, a confused model could keep calling tools forever, and every turn costs money. `track()` stops the notebook anyway if spending passes the budget.

## 4. Adding a tool: the four-step recipe (paid)

With the loop in place, adding a tool never touches the loop. Every new tool takes the same four steps:

1. **Write the function.**
2. **Write its schema.**
3. **Add the schema to the `tools` list.**
4. **Add a branch to `run_tool`.**

We add `set_reminder`. There's no real reminder service here: reminders go into a list, so you can check what Claude stored.

In [14]:
reminders = []


# Step 1: the function.
def set_reminder(content, timestamp):
    reminders.append({"content": content, "timestamp": timestamp})
    return f"Reminder set: {content!r} at {timestamp}"


# Step 2: the schema.
set_reminder_schema = {
    "name": "set_reminder",
    "description": "Set a reminder for the user at a specific date and time. "
                   "If the date has to be calculated, use add_duration_to_datetime first.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {"type": "string", "description": "What to remind the user about, e.g. 'Doctor appointment'"},
            "timestamp": {"type": "string", "description": "When to remind them, as 'YYYY-MM-DD HH:MM:SS'"},
        },
        "required": ["content", "timestamp"],
    },
}

# Step 3: add the schema to the list.
tools = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]


# Step 4: add a branch to the dispatcher.
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    if tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    if tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


messages = []
add_user_message(messages, "Set a reminder for my doctor's appointment. It's 177 days after January 1st, 2050.")
run_conversation(messages, tools)

print("\nStored reminders:", reminders)
print("Checked by our own code: 177 days after Jan 1, 2050 is", add_duration_to_datetime("2050-01-01 00:00:00", 177, "days"))

assert len(reminders) == 1
assert reminders[0]["timestamp"].startswith("2050-06-27")

[input=891 output=113 cost=$0.00291 | spent $0.0135 of $0.25]
Turn 1: stop_reason=tool_use, tools requested: ['add_duration_to_datetime']
      ran add_duration_to_datetime({'datetime_str': '2050-01-01 00:00:00', 'duration': 177, 'unit': 'days'}) -> "Monday, June 27, 2050"
[input=1,022 output=91 cost=$0.00295 | spent $0.0164 of $0.25]
Turn 2: stop_reason=tool_use, tools requested: ['set_reminder']
      ran set_reminder({'content': "Doctor's appointment", 'timestamp': '2050-06-27 09:00:00'}) -> "Reminder set: \"Doctor's appointment\" at 2050-06-27 09:00:00"
[input=1,152 output=88 cost=$0.00318 | spent $0.0196 of $0.25]
Turn 3: stop_reason=end_turn, tools requested: none

Final answer: I set a reminder for your doctor's appointment on Monday, June 27, 2050, which is 177 days after January 1st, 2050. You didn't give a time, so I set it for 9:00 AM. If your appointment is at a different time, tell me and I'll change it.

Stored reminders: [{'content': "Doctor's appointment", 'timestamp': 

**What to notice**

- `run_tools` and `run_conversation` find `run_tool` by name when they run, so they picked up the new version **without any change to the loop**.
- Claude **skipped** `get_current_datetime`, because the start date was given. Tools are available, not compulsory.

Here's the whole conversation the loop built:

In [15]:
for number, message in enumerate(messages):
    content = message["content"]
    if isinstance(content, str):
        print(f"{number}. {message['role']:<10}text: {content}")
        continue
    print(f"{number}. {message['role']}")
    for block in content:
        kind = block["type"] if isinstance(block, dict) else block.type
        if kind == "text":
            print(f"      text        {block.text.strip()[:80]!r}")
        elif kind == "tool_use":
            print(f"      tool_use    {block.name}({block.input})  id={block.id}")
        elif kind == "tool_result":
            print(f"      tool_result {block['content']}  answers id={block['tool_use_id']}")
        else:
            print(f"      {kind}")

0. user      text: Set a reminder for my doctor's appointment. It's 177 days after January 1st, 2050.
1. assistant
      tool_use    add_duration_to_datetime({'datetime_str': '2050-01-01 00:00:00', 'duration': 177, 'unit': 'days'})  id=toolu_016PoaFHG1eB5TkSiMK1Zomi
2. user
      tool_result "Monday, June 27, 2050"  answers id=toolu_016PoaFHG1eB5TkSiMK1Zomi
3. assistant
      tool_use    set_reminder({'content': "Doctor's appointment", 'timestamp': '2050-06-27 09:00:00'})  id=toolu_011PFiYxKSGHSoGcxo9MDe14
4. user
      tool_result "Reminder set: \"Doctor's appointment\" at 2050-06-27 09:00:00"  answers id=toolu_011PFiYxKSGHSoGcxo9MDe14
5. assistant
      text        "I set a reminder for your doctor's appointment on Monday, June 27, 2050, which i"


## 5. Making Claude use a tool (free)

`tool_choice` controls whether Claude uses tools:

| `tool_choice` | Meaning | On Sonnet 5.5 |
|---|---|---|
| `{"type": "auto"}` | Claude decides. This is the default | Works |
| `{"type": "none"}` | Claude may not use tools | Works |
| `{"type": "any"}` | Claude must use some tool | **400 error** |
| `{"type": "tool", "name": ...}` | Claude must use that exact tool | **400 error** |

Older tutorials force a tool with `any` or `tool`. **Sonnet 5.5 rejects both.** A rejected request generates nothing, so this cell is free.

In [16]:
import anthropic

try:
    client.messages.create(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": "What is the date today?"}],
        tools=tools,
        tool_choice={"type": "tool", "name": "get_current_datetime"},
    )
    raise AssertionError("Expected a BadRequestError")
except anthropic.BadRequestError as error:
    print(f"BadRequestError, status {error.status_code}")
    print(error.message)

BadRequestError, status 400
Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'tool_choice: type "tool" and "any" are not supported for this model.'}, 'request_id': 'req_011CgxR3cEihtgk1PzAHhni6'}


**What to do instead:**

- **Say it in the prompt:** "Use the `get_current_datetime` tool to answer." Then check the response really contains a `tool_use` block, and re-prompt if it doesn't.
- **Write descriptions that say when to call a tool,** as ours do: *"Use this whenever the answer depends on today's date."*
- **If you forced a tool only to get JSON back,** use structured outputs instead, from notebook 02.

## 6. Streaming tool arguments (paid)

When you stream a response that calls a tool, the tool's arguments arrive as `input_json` events. Each one carries:

- `partial_json`: the new piece of JSON just received,
- `snapshot`: everything received so far, parsed.

**By default the API buffers each top-level argument**, checks it against your schema, then sends it in one burst. With a tool whose input is `{"abstract": ..., "meta": {...}}`, you see a pause, a burst for `abstract`, another pause, and a burst for `meta`.

**`eager_input_streaming: True`** on a tool definition turns that buffering off. Pieces arrive as soon as Claude writes them, but **nothing checks the JSON any more**, so your code must.

> **Different from the course:** the course calls this *fine-grained tool calling* and turns it on through its own `fine_grained=True` helper and a beta header. Today it's the `eager_input_streaming` field on the tool, with no beta header.

The helper below streams one response and records when each piece of JSON arrived.

In [17]:
def save_article(abstract, meta):
    return f"Article saved: {len(abstract.split())} words, meta={meta}"


save_article_schema = {
    "name": "save_article",
    "description": "Save a research article's abstract together with metadata about it.",
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {"type": "string", "description": "The full abstract text"},
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {"type": "integer", "description": "Number of words in the abstract"},
                    "review": {"type": "string", "description": "A one-sentence review of the abstract"},
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
}

ARTICLE_PROMPT = ("Write a 100-word abstract for a research paper about quantum networking, "
                  "then save it with the save_article tool.")


def stream_tool_call(tool, eager=False):
    """Stream one response. Return (final message or None, list of (seconds, json_piece))."""
    if eager:
        tool = {**tool, "eager_input_streaming": True}   # a copy: the original stays unchanged
    pieces = []
    start = time.perf_counter()
    try:
        with client.messages.stream(
            model=MODEL,
            max_tokens=2000,
            output_config={"effort": "low"},
            messages=[{"role": "user", "content": ARTICLE_PROMPT}],
            tools=[tool],
        ) as stream:
            for event in stream:
                if event.type == "input_json":
                    pieces.append((time.perf_counter() - start, event.partial_json))
            final = stream.get_final_message()
    except ValueError as error:
        # The SDK raises ValueError for streamed JSON it can't parse at all. That can only
        # happen with eager streaming, because nothing checked the JSON on the way.
        print(f"Invalid JSON from the stream: {str(error)[:100]}")
        return None, pieces
    track(final)
    return final, pieces


def describe(pieces):
    times = [seconds for seconds, piece in pieces if piece]
    gaps = [b - a for a, b in zip(times, times[1:])]
    distinct_moments = len({round(t, 2) for t in times})
    return (f"{len(times)} pieces, first after {times[0]:.2f}s, longest gap {max(gaps, default=0):.2f}s, "
            f"arriving at {distinct_moments} distinct moments") if times else "no pieces"


buffered_message, buffered_pieces = stream_tool_call(save_article_schema)
print("Buffered (default):", describe(buffered_pieces), "\n")

eager_message, eager_pieces = stream_tool_call(save_article_schema, eager=True)
print("Eager             :", describe(eager_pieces))

[input=514 output=1,388 cost=$0.01491 | spent $0.0345 of $0.25]
Buffered (default): 149 pieces, first after 10.14s, longest gap 0.00s, arriving at 2 distinct moments 

[input=514 output=1,343 cost=$0.01446 | spent $0.0490 of $0.25]
Eager             : 47 pieces, first after 8.04s, longest gap 0.72s, arriving at 4 distinct moments


**What to notice**

- **Buffered** pieces arrive in bursts, so many pieces share the same moment. In buffered mode nothing arrives until the whole `abstract` is written and checked, so the first piece comes later.
- **Eager** starts sooner and spreads the pieces over more moments, but it can still bunch them. When I ran it, buffered arrived at 2 moments after 10.1 seconds, and eager arrived at 4 moments after 8.0 seconds. With `strict: True` added (below), it arrived at 40 separate moments. Your numbers will differ from run to run.

### Check tool input before you run the tool

Whatever the mode, **validate a tool's input before running it**. With eager streaming, the SDK's forgiving parser can even return a silently cut-off object instead of raising an error, so an exception isn't a reliable signal.

In [18]:
def validate_article_input(tool_input):
    """Return a list of problems with save_article's input. An empty list means it's valid."""
    problems = []
    if set(tool_input) != {"abstract", "meta"}:
        problems.append(f"top-level keys are {sorted(tool_input)}")
    if not isinstance(tool_input.get("abstract"), str):
        problems.append("'abstract' must be a string")
    meta = tool_input.get("meta")
    if not isinstance(meta, dict):
        problems.append(f"'meta' must be an object, got {type(meta).__name__}")
    else:
        word_count = meta.get("word_count")
        if not isinstance(word_count, int) or isinstance(word_count, bool):  # bool is a kind of int in Python
            problems.append(f"'meta.word_count' must be a whole number, got {word_count!r}")
        if not isinstance(meta.get("review"), str):
            problems.append("'meta.review' must be a string")
    return problems


for label, message in [("Buffered", buffered_message), ("Eager", eager_message)]:
    if message is None:
        print(f"{label}: no tool input, the stream failed")
        continue
    if message.stop_reason in ("max_tokens", "refusal"):
        print(f"{label}: stop_reason={message.stop_reason}, so the input may be cut off. Not running the tool.")
        continue
    tool_use = next(block for block in message.content if block.type == "tool_use")
    problems = validate_article_input(tool_use.input)
    if problems:
        print(f"{label}: INVALID, not running the tool: {problems}")
    else:
        print(f"{label}: valid. {save_article(**tool_use.input)}")

Buffered: valid. Article saved: 100 words, meta={'word_count': 100, 'review': 'A concise, well-structured abstract that states the problem, method, and results of an adaptive entanglement-distribution approach for quantum networks.'}
Eager: valid. Article saved: 100 words, meta={'word_count': 100, 'review': 'A concise, well-structured abstract that states the problem, proposed hybrid architecture and routing method, quantitative simulation results, and the main conclusion.'}


**If the input is invalid,** don't run the tool. Send a `tool_result` with `"is_error": True` listing the problems, and Claude will usually call the tool again with corrected input. If the stream itself failed, there's no `tool_use_id` to reply to, so send the request again, up to a small limit.

### The strongest fix: `strict: True`

With `"strict": True` on a tool, the API **guarantees** the input matches your schema, and that still holds with eager streaming. It needs `"additionalProperties": False` on every object, and every field listed in `required`.

In [19]:
import copy

strict_schema = copy.deepcopy(save_article_schema)
strict_schema["strict"] = True
strict_schema["input_schema"]["additionalProperties"] = False
strict_schema["input_schema"]["properties"]["meta"]["additionalProperties"] = False

strict_message, strict_pieces = stream_tool_call(strict_schema, eager=True)
print("Eager + strict:", describe(strict_pieces))

assert strict_message is not None
tool_use = next(block for block in strict_message.content if block.type == "tool_use")
assert validate_article_input(tool_use.input) == []
print("Valid:", save_article(**tool_use.input))

[input=534 output=1,119 cost=$0.01226 | spent $0.0612 of $0.25]
Eager + strict: 40 pieces, first after 8.46s, longest gap 0.11s, arriving at 40 distinct moments
Valid: Article saved: 100 words, meta={'word_count': 100, 'review': 'A concise, well-structured abstract covering motivation, method, results, and implications of a modular quantum network architecture.'}


| | Buffered (default) | Eager | Eager + strict |
|---|---|---|---|
| How pieces arrive | Bursts, one per top-level argument | Steadily, as written | Steadily, as written |
| Schema checked by the API | Yes, but a bad value can still come back as the wrong type | No | **Guaranteed** |
| Your code must | Validate the input | Catch errors, re-send, and validate | Nothing extra |

## 7. Letting the SDK run the loop (paid)

You've now written the tool loop by hand, so you know what it does. In real projects, the SDK can run it for you with its **tool runner** (currently in beta):

- Decorate each function with `@beta_tool`. **The schema is built from the type hints and docstring**, so you don't write it.
- `client.beta.messages.tool_runner(...)` calls Claude, runs the requested tools, sends the results back, and repeats until Claude answers.
- Each loop step gives you the message, so you can still log, check and track spending as you go.

In [20]:
from anthropic import beta_tool


@beta_tool
def get_today() -> str:
    """Get the current date and time. Use this whenever the answer depends on today's date."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@beta_tool
def add_days(datetime_str: str, days: int) -> str:
    """Add a number of days to a date and time, and return the resulting weekday and date.

    Args:
        datetime_str: The starting date and time, as 'YYYY-MM-DD HH:MM:SS'.
        days: How many days to add.
    """
    start = datetime.strptime(datetime_str, "%Y-%m-%d %H:%M:%S")
    return (start + timedelta(days=days)).strftime("%A, %B %d, %Y")


runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    tools=[get_today, add_days],
    max_iterations=5,
    messages=[{"role": "user", "content": "What day of the week will it be 11 days from today?"}],
)

for step, message in enumerate(runner, start=1):
    track(message)
    requested = [block.name for block in message.content if block.type == "tool_use"]
    print(f"Step {step}: stop_reason={message.stop_reason}, tools requested: {requested or 'none'}")

print("\nFinal answer:", text_from_message(message))
assert message.stop_reason == "end_turn"

[input=568 output=26 cost=$0.00140 | spent $0.0626 of $0.25]
Step 1: stop_reason=tool_use, tools requested: ['get_today']
[input=611 output=83 cost=$0.00205 | spent $0.0647 of $0.25]
Step 2: stop_reason=tool_use, tools requested: ['add_days']
[input=710 output=35 cost=$0.00177 | spent $0.0664 of $0.25]
Step 3: stop_reason=end_turn, tools requested: none

Final answer: 11 days from today (November 12, 2026) will be a **Monday**, November 23, 2026.


**What to notice**

- Compare the length of this cell with sections 3 and 4. No schemas, no dispatcher, no loop.
- The same steps still happen. Knowing what the runner does for you is why we wrote it by hand first.
- Be careful with tools that change things, such as sending email or deleting data. The runner runs them automatically, so put checks inside the tool function itself, or keep a manual loop where a person approves each call.

## Final spend

In [21]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

This notebook spent $0.0664 of its $0.25 budget.


## Takeaways

1. **A tool is a function plus a schema.** Claude sees only the schema, so write a description that says when to use it.
2. **Claude asks, and your code runs.** A `tool_use` block is a request, and you reply with a `tool_result`.
3. **Match results to requests by `id`**, and return every result in one user message.
4. **Loop while `stop_reason` is `tool_use`**, with a turn limit.
5. **Adding a tool never changes the loop:** function, schema, list, dispatcher.
6. **Forced `tool_choice` is gone on Sonnet 5.5.** Ask in the prompt, and use structured outputs when you only need JSON.
7. **Validate tool input before running it**, or use `strict: True` to have it guaranteed.
8. **The SDK's tool runner** handles the loop once you understand it.

## Your turn

Each exercise costs a cent or two.

- Add a fourth tool, `list_reminders`, which returns the `reminders` list, using the four-step recipe. Ask Claude what reminders you have.
- Make `add_duration_to_datetime` raise an error for durations over 100 years, then ask for a reminder 200 years from now. How does Claude react to the `is_error` result?
- In section 6, add `"keywords": {"type": "array", "items": {"type": "string"}}` to `save_article_schema` and update `validate_article_input`.
- In section 7, add `set_reminder` as a `@beta_tool` and repeat the doctor's appointment question.

**Next:** [06 · Built-in tools](06_built_in_tools.ipynb): tools that Anthropic defines, including a text editor, web search and code execution.